In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path

PROJECT_ROOT = Path('..')

# Validação da Co-simulação de Veículos Elétricos
Este notebook executa o cenário de co-simulação `scenario_ESB01S4_EVs.py` e realiza a análise do comportamento dinâmico dos carregadores (EVSEs) acoplados fisicamente à rede elétrica via Mosaik e Caldera ICM.

In [19]:
# Carregando os dois cenários: COM e SEM EVs
df_evs = pd.read_csv('../output/result_run_ESB01S4_EVs.csv')
df_evs['date'] = pd.to_datetime(df_evs['date'])
df_evs.set_index('date', inplace=True)

df_base = pd.read_csv('../output/result_run_ESB01S4.csv')
df_base['date'] = pd.to_datetime(df_base['date'])
df_base.set_index('date', inplace=True)

print(f"Linhas Cenário Base: {len(df_base)} | Linhas Cenário EVs: {len(df_evs)}")


# Comparação de Tensão Média nas Barras de Baixa Tensão (BT)
import numpy as np

def get_mean_voltage(df_scenario):
    v1_cols = [c for c in df_scenario.columns if 'V1_pu' in c]
    v2_cols = [c for c in df_scenario.columns if 'V2_pu' in c]
    v3_cols = [c for c in df_scenario.columns if 'V3_pu' in c]
    
    df_v1 = df_scenario[v1_cols].replace([np.inf, -np.inf], np.nan)
    df_v2 = df_scenario[v2_cols].replace([np.inf, -np.inf], np.nan)
    df_v3 = df_scenario[v3_cols].replace([np.inf, -np.inf], np.nan)
    
    return (
        df_v1[df_v1 > 0.1].mean(axis=1),
        df_v2[df_v2 > 0.1].mean(axis=1),
        df_v3[df_v3 > 0.1].mean(axis=1)
    )

v1_evs, v2_evs, v3_evs = get_mean_voltage(df_evs)
v1_base, v2_base, v3_base = get_mean_voltage(df_base)

fig, ax = plt.subplots(figsize=(14, 5))

# Cenário Base (Pontilhado)
ax.plot(df_base.index, v1_base, color='red', linestyle='--', linewidth=1.5, alpha=0.5, label='Fase A (Sem EV)')
ax.plot(df_base.index, v2_base, color='green', linestyle='--', linewidth=1.5, alpha=0.5, label='Fase B (Sem EV)')
ax.plot(df_base.index, v3_base, color='blue', linestyle='--', linewidth=1.5, alpha=0.5, label='Fase C (Sem EV)')

# Cenário com EVs (Sólido)
ax.plot(df_evs.index, v1_evs, color='red', linewidth=2.0, label='Fase A (Com EV)')
ax.plot(df_evs.index, v2_evs, color='green', linewidth=2.0, label='Fase B (Com EV)')
ax.plot(df_evs.index, v3_evs, color='blue', linewidth=2.0, label='Fase C (Com EV)')

ax.set_title('Impacto dos EVs na Tensão Média (Baixa Tensão)', fontsize=14, fontweight='bold')
ax.set_ylabel('Tensão Média (pu)', fontsize=12)
ax.set_xlabel('Tempo', fontsize=12)
ax.grid(True, linestyle='--', alpha=0.6)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%a, %d/%m'))
plt.xticks(rotation=45)
ax.legend(ncol=2)
plt.tight_layout()
plt.show()


In [20]:
# Potência Média dos EVs que estão carregando no momento
ev_cols = [c for c in df_evs.columns if 'P_kw' in c and 'Caldera' in c]

if len(ev_cols) > 0:
    ev_active = df_evs[ev_cols][df_evs[ev_cols] > 0.1]
    ev_mean_power = ev_active.mean(axis=1).fillna(0)
    
    fig, ax = plt.subplots(figsize=(14, 5))
    
    ax.plot(df_evs.index, ev_mean_power, color='darkorange', linewidth=2)
    ax.fill_between(df_evs.index, ev_mean_power, alpha=0.3, color='orange')
    
    ax.set_title('Potência Média (apenas VEs ativos/carregando)', fontsize=14, fontweight='bold')
    ax.set_ylabel('Potência Ativa Média (kW)', fontsize=12)
    ax.set_xlabel('Tempo', fontsize=12)
    ax.grid(True, linestyle='--', alpha=0.6)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%a, %d/%m'))
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("Colunas de EVs não encontradas no dataset atual.")


## 2. Leitura e Processamento dos Resultados

In [21]:
CSV_FILE = PROJECT_ROOT / 'output' / 'result_run_ESB01S4_EVs.csv'

print(f"Lendo resultados do arquivo: {CSV_FILE.name}...")
df = pd.read_csv(CSV_FILE)
df['date'] = pd.to_datetime(df['date'])
df.set_index('date', inplace=True)

print(f"Linhas totais: {len(df)}")
print(f"Número de colunas coletadas: {len(df.columns)}")

## 3. Relatório do Terminal: Demanda Ativa e Fator de Potência dos VEs

In [22]:
# Filtrando as colunas relacionadas a Veículos Elétricos
ev_p_cols = [c for c in df.columns if 'CalderaEV' in c and 'P_kw' in c]
ev_q_cols = [c for c in df.columns if 'CalderaEV' in c and 'Q_kvar' in c]
ev_fp_cols = [c for c in df.columns if 'CalderaEV' in c and 'fp' in c]

df['Total_EV_P_kw'] = df[ev_p_cols].sum(axis=1)
df['Total_EV_Q_kvar'] = df[ev_q_cols].sum(axis=1)

max_p_kw = df['Total_EV_P_kw'].max()
time_max_p = df['Total_EV_P_kw'].idxmax()

step_size_h = 1.0 / 60.0 # 1 minuto = 1/60 horas
energia_total_kwh = df['Total_EV_P_kw'].sum() * step_size_h

print("="*60)
print("           RELATÓRIO DE COMPORTAMENTO DOS VEs")
print("="*60)
print(f"Total de VEs simulados: {len(ev_p_cols)}")
print(f"Energia total demandada na semana: {energia_total_kwh:.2f} kWh")
print(f"Pico de Demanda Agregada dos VEs: {max_p_kw:.2f} kW")
print(f"Ocorrência do Pico: {time_max_p.strftime('%d/%m/%Y %H:%M')}")

# Analisando o Fator de Potência (PFC)
mean_fps = df[ev_fp_cols].replace(0, np.nan).mean().mean()
print(f"Fator de Potência médio em operação (Active PFC): {mean_fps:.4f}")
print("="*60)

## 4. Análise Gráfica

In [23]:
plt.style.use('default')
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(df.index, df['Total_EV_P_kw'], color='#FF5722', linewidth=1.5, label='Demanda Ativa VEs (P_kw)')
ax.plot(df.index, df['Total_EV_Q_kvar'], color='#03A9F4', linewidth=1.5, label='Demanda Reativa VEs (Q_kvar)', alpha=0.8)

ax.set_title('Perfil de Demanda Agregada de Todos os Veículos Elétricos', fontsize=14, fontweight='bold')
ax.set_ylabel('Potência (kW / kVAr)', fontsize=12)
ax.set_xlabel('Tempo', fontsize=12)
ax.legend(loc='upper right')
ax.grid(True, linestyle='--', alpha=0.6)

# Formatando eixo x
ax.xaxis.set_major_locator(mdates.DayLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%a, %d/%m'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [24]:
fig, ax = plt.subplots(figsize=(14, 5))

# Selecionar os 3 VEs que consumiram mais energia para evidenciar o Tapering (CC-CV)
top_3_evs = df[ev_p_cols].sum().nlargest(3).index

for i, col in enumerate(top_3_evs):
    ax.plot(df.index, df[col], linewidth=2, label=f'Veículo {i+1} (Wallbox)')

ax.set_title('Assinatura Física Individual (Evidência do Efeito Tapering CV)', fontsize=14, fontweight='bold')
ax.set_ylabel('Potência Ativa (kW)', fontsize=12)
ax.set_xlabel('Tempo', fontsize=12)
ax.legend()
ax.grid(True, linestyle='--', alpha=0.6)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%a %H:%M'))
plt.xticks(rotation=45)

# Dando zoom nas primeiras 72 horas para ficar mais visível
ax.set_xlim(df.index[0], df.index[0] + pd.Timedelta(hours=72))

plt.tight_layout()
plt.show()

In [25]:
# Tensão Média nas Barras de Baixa Tensão (BT) por Fase
v1_cols = [c for c in df.columns if 'V1_pu' in c]
v2_cols = [c for c in df.columns if 'V2_pu' in c]
v3_cols = [c for c in df.columns if 'V3_pu' in c]

if len(v1_cols) > 0:
    # Filtrando as colunas ignorando valores < 0.1 (fases inexistentes que retornam 2e-8)
    # Além de ignorar os infinitos que destruíam o gráfico antes da correção do modelo
    import numpy as np
    df_v1 = df[v1_cols].replace([np.inf, -np.inf], np.nan)
    df_v2 = df[v2_cols].replace([np.inf, -np.inf], np.nan)
    df_v3 = df[v3_cols].replace([np.inf, -np.inf], np.nan)
    
    v1_mean = df_v1[df_v1 > 0.1].mean(axis=1)
    v2_mean = df_v2[df_v2 > 0.1].mean(axis=1)
    v3_mean = df_v3[df_v3 > 0.1].mean(axis=1)
    
    fig, ax = plt.subplots(figsize=(14, 5))
    
    ax.plot(df.index, v1_mean, color='red', linewidth=1.5, label='Fase A')
    ax.plot(df.index, v2_mean, color='green', linewidth=1.5, label='Fase B')
    ax.plot(df.index, v3_mean, color='blue', linewidth=1.5, label='Fase C')
    
    ax.set_title('Tensão Média nas Barras de Baixa Tensão (Fases A, B e C)', fontsize=14, fontweight='bold')
    ax.set_ylabel('Tensão (pu)', fontsize=12)
    ax.set_xlabel('Tempo', fontsize=12)
    ax.grid(True, linestyle='--', alpha=0.6)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%a, %d/%m'))
    plt.xticks(rotation=45)
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Colunas de Tensão não encontradas no dataset atual.")


In [26]:
# Potência Média dos EVs que estão carregando no momento
ev_cols = [c for c in df.columns if 'P_kw' in c and 'Caldera' in c]

if len(ev_cols) > 0:
    # Pegamos apenas os EVs que estão puxando mais de 0.1 kW (ignorando os parados)
    ev_active = df[ev_cols][df[ev_cols] > 0.1]
    ev_mean_power = ev_active.mean(axis=1).fillna(0)
    
    fig, ax = plt.subplots(figsize=(14, 5))
    
    ax.plot(df.index, ev_mean_power, color='darkorange', linewidth=2)
    ax.fill_between(df.index, ev_mean_power, alpha=0.3, color='orange')
    
    ax.set_title('Potência Média (apenas VEs ativos/carregando)', fontsize=14, fontweight='bold')
    ax.set_ylabel('Potência Ativa Média (kW)', fontsize=12)
    ax.set_xlabel('Tempo', fontsize=12)
    ax.grid(True, linestyle='--', alpha=0.6)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%a, %d/%m'))
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("Colunas de EVs não encontradas no dataset atual.")
